In [26]:
import os
import sys

PROJECT_ROOT = os.path.abspath("..")
sys.path.append(PROJECT_ROOT)

%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [37]:
import os
import sys

import numpy as np
import pandas as pd

from sklearn.model_selection import KFold, cross_val_score
from sklearn.pipeline import Pipeline

from src.preprocessing import get_preprocessor, build_preprocessor

from lightgbm import LGBMRegressor

In [5]:
train = pd.read_csv('../data/train.csv')

X = train.drop(columns='SalePrice')
y = train['SalePrice']

y_log = np.log1p(y)

print(X.shape)
print(y_log.shape)

(1460, 80)
(1460,)


In [ ]:
SEED = 42

cv = KFold(
    n_splits=5,
    shuffle=True,
    random_state=SEED,
)

In [9]:
def evaluate_lgbm(X, y, cv):
    model = Pipeline([
        ('preprocessor', get_preprocessor(X)),
        ('model', LGBMRegressor(
                    n_estimators=500,
                    learning_rate=0.03,
                    random_state=SEED,
                    verbosity=-1))
        ])
            
    scores = cross_val_score(
        model, 
        X,
        y,
        cv=cv,
        scoring="neg_root_mean_squared_error",
        n_jobs=-1,
    )
    
    rmse_scores = -scores

    return {
        "fold_rmse": rmse_scores,
        "mean_rmse": rmse_scores.mean(),
        "std_rmse": rmse_scores.std(),
    }
    

In [12]:
baseline_result = evaluate_lgbm(X, y_log, cv)

print("Fold RMSE:", np.round(baseline_result["fold_rmse"], 6))

print(f"Mean RMSE: "f"{baseline_result['mean_rmse']:.6f}")

print(f"Std RMSE: "f"{baseline_result['std_rmse']:.6f}")

Fold RMSE: [0.138282 0.119627 0.166385 0.133824 0.115829]
Mean RMSE: 0.134789
Std RMSE: 0.017893


In [13]:
X_fe = X.copy()

X_fe['TotalSF'] = (
    X_fe['TotalBsmtSF']
    + X_fe['1stFlrSF']
    + X_fe['2ndFlrSF']
)

total_sf_result = evaluate_lgbm(
    X_fe,
    y_log,
    cv
)

print("Fold RMSE:", np.round(total_sf_result["fold_rmse"], 6))

print(f"Mean RMSE: "f"{total_sf_result['mean_rmse']:.6f}")

print(f"Std RMSE: "f"{total_sf_result['std_rmse']:.6f}")

improvement = (
    baseline_result["mean_rmse"]
    - total_sf_result["mean_rmse"]
)

print(f"Improvement: {improvement:.6f}")

Fold RMSE: [0.138604 0.119152 0.163659 0.1303   0.11347 ]
Mean RMSE: 0.133037
Std RMSE: 0.017617
Improvement: 0.001752


In [22]:
X_fe = X.copy()

X_fe['TotalSF'] = (
    X_fe['TotalBsmtSF']
    + X_fe['1stFlrSF']
    + X_fe['2ndFlrSF']
)

X_fe['HouseAge'] = (
    X_fe['YrSold']
    - X_fe['YearBuilt']
)

house_age_result = evaluate_lgbm(
    X_fe,
    y_log,
    cv
)

print("Fold RMSE:", np.round(house_age_result["fold_rmse"], 6))

print(f"Mean RMSE: "f"{house_age_result['mean_rmse']:.6f}")

print(f"Std RMSE: "f"{house_age_result['std_rmse']:.6f}")

improvement = (
    baseline_result["mean_rmse"]
    - house_age_result["mean_rmse"]
)

print(f"Improvement: {improvement:.6f}")

Fold RMSE: [0.139213 0.118084 0.165671 0.131575 0.113969]
Mean RMSE: 0.133702
Std RMSE: 0.018386
Improvement: 0.001087


In [16]:
X_fe = X.copy()

X_fe['TotalSF'] = (
    X_fe['TotalBsmtSF']
    + X_fe['1stFlrSF']
    + X_fe['2ndFlrSF']
)

X_fe["RemodAge"] = (
    X_fe["YrSold"]
    - X_fe["YearRemodAdd"]
)

remod_age_result = evaluate_lgbm(
    X_fe,
    y_log,
    cv
)

print("Fold RMSE:", np.round(remod_age_result["fold_rmse"], 6))

print(f"Mean RMSE: "f"{remod_age_result['mean_rmse']:.6f}")

print(f"Std RMSE: "f"{remod_age_result['std_rmse']:.6f}")

improvement = (
    baseline_result["mean_rmse"]
    - remod_age_result["mean_rmse"]
)

print(f"Improvement: {improvement:.6f}")

Fold RMSE: [0.139075 0.11735  0.164899 0.129901 0.113821]
Mean RMSE: 0.133009
Std RMSE: 0.018313
Improvement: 0.001780


In [17]:
X_fe = X.copy()

X_fe['TotalSF'] = (
    X_fe['TotalBsmtSF']
    + X_fe['1stFlrSF']
    + X_fe['2ndFlrSF']
)

X_fe["TotalBathrooms"] = (
    X_fe["FullBath"]
    + 0.5 * X_fe["HalfBath"]
    + X_fe["BsmtFullBath"].fillna(0)
    + 0.5 * X_fe["BsmtHalfBath"].fillna(0)
)

bathrooms_result = evaluate_lgbm(
    X_fe,
    y_log,
    cv
)

print("Fold RMSE:", np.round(bathrooms_result["fold_rmse"], 6))

print(f"Mean RMSE: "f"{bathrooms_result['mean_rmse']:.6f}")

print(f"Std RMSE: "f"{bathrooms_result['std_rmse']:.6f}")

improvement = (
    total_sf_result["mean_rmse"]
    - bathrooms_result["mean_rmse"]
)

print(f"Improvement: {improvement:.6f}")

Fold RMSE: [0.138584 0.118591 0.163809 0.129748 0.112738]
Mean RMSE: 0.132694
Std RMSE: 0.017937
Improvement: 0.000343


In [21]:
X_fe = X.copy()

X_fe['TotalSF'] = (
    X_fe['TotalBsmtSF']
    + X_fe['1stFlrSF']
    + X_fe['2ndFlrSF']
)

X_fe["TotalBathrooms"] = (
    X_fe["FullBath"]
    + 0.5 * X_fe["HalfBath"]
    + X_fe["BsmtFullBath"].fillna(0)
    + 0.5 * X_fe["BsmtHalfBath"].fillna(0)
)

X_fe["TotalPorchSF"] = (
    X_fe["OpenPorchSF"]
    + X_fe["EnclosedPorch"]
    + X_fe["3SsnPorch"]
    + X_fe["ScreenPorch"]
    + X_fe["WoodDeckSF"]
)

porch_result = evaluate_lgbm(
    X_fe,
    y_log,
    cv
)

print("Fold RMSE:", np.round(porch_result["fold_rmse"], 6))

print(f"Mean RMSE: "f"{porch_result['mean_rmse']:.6f}")

print(f"Std RMSE: "f"{porch_result['std_rmse']:.6f}")

improvement = (
    bathrooms_result["mean_rmse"]
    - porch_result["mean_rmse"]
)

print(f"Improvement: {improvement:.6f}")

Fold RMSE: [0.139995 0.11911  0.164386 0.129106 0.111703]
Mean RMSE: 0.132860
Std RMSE: 0.018415
Improvement: -0.000166


In [20]:
X_fe = X.copy()

X_fe['TotalSF'] = (
    X_fe['TotalBsmtSF']
    + X_fe['1stFlrSF']
    + X_fe['2ndFlrSF']
)

X_fe["TotalBathrooms"] = (
    X_fe["FullBath"]
    + 0.5 * X_fe["HalfBath"]
    + X_fe["BsmtFullBath"].fillna(0)
    + 0.5 * X_fe["BsmtHalfBath"].fillna(0)
)

X_fe["OverallQual_GrLivArea"] = (
    X_fe["OverallQual"]
    * X_fe["GrLivArea"]
)

interaction_result = evaluate_lgbm(
    X_fe,
    y_log,
    cv
)

print("Fold RMSE:", np.round(interaction_result["fold_rmse"], 6))

print(f"Mean RMSE: "f"{interaction_result['mean_rmse']:.6f}")

print(f"Std RMSE: "f"{interaction_result['std_rmse']:.6f}")

improvement = (
    bathrooms_result["mean_rmse"]
    - interaction_result["mean_rmse"]
)

print(f"Improvement: {improvement:.6f}")

Fold RMSE: [0.140778 0.121894 0.16264  0.13648  0.107914]
Mean RMSE: 0.133941
Std RMSE: 0.018436
Improvement: -0.001247


In [ ]:
preprocessor = build_preprocessor(
    X,
    model_type='catboost',
    use_total_sf=True,
    use_total_bathrooms=True,
    use_house_age=True,
)

X_processed = preprocessor.fit_transform(X)

print(X.shape)
print(X_processed.shape)

X_processed_df = pd.DataFrame(X_processed)

print(X_processed_df.isna().sum().sum())

(1460, 80)
(1460, 83)
0
